<a href="https://colab.research.google.com/github/jennarandall785/Portfolio/blob/main/student_dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-learn==1.5.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 27.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [54]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

url = "https://raw.githubusercontent.com/jennarandall785/Portfolio/refs/heads/main/student_dropout_dataset.csv"

student_dropout = pd.read_csv(url)

student_dropout.head()

,student_id,age,region,enroll_date,exam_season,courses_enrolled,completed_assignments,completion_rate,login_frequency,last_activity_days_ago,forum_posts_count,dropout_score,label,label_multiclass,label_name
0,STU00001,26,Alexandria,2024-01-13,0,3,5,0.3571,5.29,10,5,0.1064,0,0,active
1,STU00002,23,Amman,2024-05-05,0,6,0,0.0000,0.84,7,0,0.6627,1,2,dropped
2,STU00003,17,Dubai,2024-03-12,0,3,1,0.0435,1.79,36,0,0.7299,1,2,dropped
3,STU00004,23,Alexandria,2024-12-12,0,6,13,0.4396,0.78,9,0,0.5315,1,1,at-risk
4,STU00005,20,Baghdad,2024-02-14,0,5,5,0.2078,0.92,11,0,0.7904,1,2,dropped


In [55]:
student_dropout['region'].value_counts()

,count
region,
Doha,534
Baghdad,532
Riyadh,510
Casablanca,509
Tunis,499
Alexandria,493
Cairo,484
Dubai,482
Amman,481


In [56]:
student_dropout['enroll_date'] = pd.to_datetime(student_dropout['enroll_date'])

student_dropout['enroll_month'] = student_dropout['enroll_date'].dt.month
student_dropout['enroll_year'] = student_dropout['enroll_date'].dt.year
student_dropout['enroll_dayofweek'] = student_dropout['enroll_date'].dt.dayofweek

In [57]:

student_dropout = pd.get_dummies(student_dropout, columns=["region"], drop_first=True)

In [58]:
X = student_dropout.drop(columns = ['student_id', 'label', 'label_multiclass', 'label_name', 'enroll_date', 'dropout_score', 'forum_posts_count'])
y = student_dropout['label']

#Create Holdout data
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X, y, test_size=0.15, random_state=42)

#Create training and vaidation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42)

In [59]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=1,
    gamma=0.1,
    early_stopping_rounds=10
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_val_pred = model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("")
print(classification_report(y_val, y_val_pred))

Validation Accuracy: 0.9197860962566845

              precision    recall  f1-score   support

           0       0.90      0.87      0.88       261
           1       0.93      0.95      0.94       487

    accuracy                           0.92       748
   macro avg       0.91      0.91      0.91       748
weighted avg       0.92      0.92      0.92       748



In [60]:
#Holdout data
y_holdout_pred = model.predict(X_holdout)
print("Holdout Accuracy:", accuracy_score(y_holdout, y_holdout_pred))
print("")
print(classification_report(y_holdout, y_holdout_pred))

Holdout Accuracy: 0.908

              precision    recall  f1-score   support

           0       0.88      0.83      0.85       243
           1       0.92      0.94      0.93       507

    accuracy                           0.91       750
   macro avg       0.90      0.89      0.89       750
weighted avg       0.91      0.91      0.91       750

